<a href="https://colab.research.google.com/github/adidror005/Sefaria/blob/main/top_torah_n_grams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests

url = "https://www.sefaria.org/api/index"
toc = requests.get(url, timeout=60).json()

In [ ]:
def collect_titles(node, keyword):
    local_results = []
    if isinstance(node, list):
        for x in node:
            local_results.extend(collect_titles(x, keyword))
    elif isinstance(node, dict):
        label = node.get("title") or node.get("category")
        if label and keyword in label:
            local_results.append(label)
        local_results.extend(collect_titles(node.get("contents", []), keyword))
    return local_results

results = collect_titles(toc, "Shemot")  # or "Genesis", etc.
results

['Gur Aryeh on Shemot',
 'Shemot Rabbah',
 'Chidushei HaRadal on Shemot Rabbah',
 'Etz Yosef on Shemot Rabbah',
 'Matnot Kehunah on Shemot Rabbah',
 'Perush Maharzu on Shemot Rabbah',
 'Yedei Moshe on Shemot Rabbah',
 "Yefeh To'ar on Shemot Rabbah",
 'Get HaShemot']

In [ ]:
import requests

ref = "Shemot"
data = requests.get(
    f"https://www.sefaria.org/api/v3/texts/{ref}"
).json()
print(data["ref"])

Exodus


In [ ]:
data['versions'][0]['text']

[['וְאֵ֗לֶּה שְׁמוֹת֙ בְּנֵ֣י יִשְׂרָאֵ֔ל הַבָּאִ֖ים מִצְרָ֑יְמָה אֵ֣ת יַעֲקֹ֔ב אִ֥ישׁ וּבֵית֖וֹ בָּֽאוּ׃',
  'רְאוּבֵ֣ן שִׁמְע֔וֹן לֵוִ֖י וִיהוּדָֽה׃',
  'יִשָּׂשכָ֥ר זְבוּלֻ֖ן וּבִנְיָמִֽן׃',
  'דָּ֥ן וְנַפְתָּלִ֖י גָּ֥ד וְאָשֵֽׁר׃',
  'וַֽיְהִ֗י כׇּל־נֶ֛פֶשׁ יֹצְאֵ֥י יֶֽרֶךְ־יַעֲקֹ֖ב שִׁבְעִ֣ים נָ֑פֶשׁ וְיוֹסֵ֖ף הָיָ֥ה בְמִצְרָֽיִם׃',
  'וַיָּ֤מׇת יוֹסֵף֙ וְכׇל־אֶחָ֔יו וְכֹ֖ל הַדּ֥וֹר הַהֽוּא׃',
  'וּבְנֵ֣י יִשְׂרָאֵ֗ל פָּר֧וּ וַֽיִּשְׁרְצ֛וּ וַיִּרְבּ֥וּ וַיַּֽעַצְמ֖וּ בִּמְאֹ֣ד מְאֹ֑ד וַתִּמָּלֵ֥א הָאָ֖רֶץ אֹתָֽם׃&nbsp;<span class="mam-spi-pe">{פ}</span><br>',
  'וַיָּ֥קׇם מֶֽלֶךְ־חָדָ֖שׁ עַל־מִצְרָ֑יִם אֲשֶׁ֥ר לֹֽא־יָדַ֖ע אֶת־יוֹסֵֽף׃',
  'וַיֹּ֖אמֶר אֶל־עַמּ֑וֹ הִנֵּ֗ה עַ֚ם בְּנֵ֣י יִשְׂרָאֵ֔ל רַ֥ב וְעָצ֖וּם מִמֶּֽנּוּ׃',
  'הָ֥בָה נִֽתְחַכְּמָ֖ה ל֑וֹ פֶּן־יִרְבֶּ֗ה וְהָיָ֞ה כִּֽי־תִקְרֶ֤אנָה מִלְחָמָה֙ וְנוֹסַ֤ף גַּם־הוּא֙ עַל־שֹׂ֣נְאֵ֔ינוּ וְנִלְחַם־בָּ֖נוּ וְעָלָ֥ה מִן־הָאָֽרֶץ׃',
  'וַיָּשִׂ֤ימוּ עָלָיו֙ שָׂרֵ֣י מִסִּ֔ים לְמַ֥עַן עַנֹּת֖וֹ בְּסִבְלֹתָ֑ם וַיִּ֜בֶן עָרֵ֤י מִסְ

In [ ]:
# Install and import necessary libraries for text processing
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from collections import Counter

# Extract the text from the 'data' variable
# The text is a nested list, so flatten it into a single string
raw_text_list = data['versions'][0]['text']

def flatten_list(nested_list):
    flat_list = []
    for item in nested_list:
        if isinstance(item, list):
            flat_list.extend(flatten_list(item))
        else:
            flat_list.append(item)
    return flat_list

flat_text = ' '.join(flatten_list(raw_text_list))

# Clean the text: remove HTML tags, special characters (except Hebrew letters and basic punctuation), and convert to lowercase
clean_text = re.sub(r'<.*?>', '', flat_text) # Remove HTML tags

# Remove Hebrew diacritics (nikkud) and other combining characters (U+0591–U+05BD, U+05BF, U+05C1–U+05C2, U+05C4–U+05C7)
clean_text = re.sub(r'[\u0591-\u05BD\u05BF\u05C1-\u05C2\u05C4-\u05C7]', '', clean_text)

# Replace any character that is NOT a Hebrew letter or space with a space (removes non-Hebrew, including English like 'nbsp')
clean_text = re.sub(r'[^א-ׯ\s]', ' ', clean_text)

# Replace multiple spaces with a single space and strip leading/trailing spaces
clean_text = re.sub(r'\s+', ' ', clean_text).strip()

clean_text = clean_text.lower() # Convert to lowercase (this will primarily affect English words if present)

print(f"clean_text snippet: {clean_text[:500]}") # Display a snippet of the cleaned text

clean_text snippet: ואלה שמות בני ישראל הבאים מצרימה את יעקב איש וביתו באו ראובן שמעון לוי ויהודה יששכר זבולן ובנימן דן ונפתלי גד ואשר ויהי כל נפש יצאי ירך יעקב שבעים נפש ויוסף היה במצרים וימת יוסף וכל אחיו וכל הדור ההוא ובני ישראל פרו וישרצו וירבו ויעצמו במאד מאד ותמלא הארץ אתם nbsp פ ויקם מלך חדש על מצרים אשר לא ידע את יוסף ויאמר אל עמו הנה עם בני ישראל רב ועצום ממנו הבה נתחכמה לו פן ירבה והיה כי תקראנה מלחמה ונוסף גם הוא על שנאינו ונלחם בנו ועלה מן הארץ וישימו עליו שרי מסים למען ענתו בסבלתם ויבן ערי מסכנות לפרעה


In [ ]:
# Tokenize the text
# Using simple split by whitespace after thorough cleaning in the previous cell
words = clean_text.split()

# Remove stopwords (using Hebrew stopwords)
# NLTK's Hebrew stopwords are not extensive, but will remove common ones if present
# If no Hebrew stopwords are found, or for more comprehensive filtering, custom lists might be needed.
try:
    stop_words = set(stopwords.words('hebrew'))
except OSError:
    print("Hebrew stopwords not found. Using an empty set for now. Consider downloading if available or creating a custom list.")
    stop_words = set()

filtered_words = [word for word in words if word not in stop_words and len(word) > 1]

print(f"Total words after cleaning and filtering: {len(filtered_words)}")
print(filtered_words[:50])

Total words after cleaning and filtering: 18284
['ואלה', 'שמות', 'בני', 'ישראל', 'הבאים', 'מצרימה', 'את', 'יעקב', 'איש', 'וביתו', 'באו', 'ראובן', 'שמעון', 'לוי', 'ויהודה', 'יששכר', 'זבולן', 'ובנימן', 'דן', 'ונפתלי', 'גד', 'ואשר', 'ויהי', 'כל', 'נפש', 'יצאי', 'ירך', 'יעקב', 'שבעים', 'נפש', 'ויוסף', 'היה', 'במצרים', 'וימת', 'יוסף', 'וכל', 'אחיו', 'וכל', 'הדור', 'ההוא', 'ובני', 'ישראל', 'פרו', 'וישרצו', 'וירבו', 'ויעצמו', 'במאד', 'מאד', 'ותמלא', 'הארץ']


Now, let's find the top N-grams. We'll start with bigrams (N=2) and you can easily change `N` to any other value you like.

In [ ]:
# Generate N-grams (e.g., bigrams, n=2)
N = 3 # Changed N to 3 for trigrams
ngram_list = list(ngrams(words, N))

# Count the frequency of each N-gram
ngram_counts = Counter(ngram_list)

# Get the top 10 most common N-grams
top_n_grams = ngram_counts.most_common(10)

print(f"Top {N}-grams:")
for ngram, count in top_n_grams:
    print(f"{ngram}: {count}")

Top 3-grams:
('יהוה', 'אל', 'משה'): 61
('ויאמר', 'יהוה', 'אל'): 43
('כאשר', 'צוה', 'יהוה'): 22
('צוה', 'יהוה', 'את'): 20
('יהוה', 'את', 'משה'): 19
('וארגמן', 'ותולעת', 'שני'): 18
('את', 'בני', 'ישראל'): 17
('פ', 'ויאמר', 'יהוה'): 17
('ויאמר', 'משה', 'אל'): 15
('אל', 'בני', 'ישראל'): 14


In [ ]:
from collections import Counter

prefixes = ("ו", "ב", "ל", "כ", "מ", "ה")

# Modified strip_prefixes to handle names that start with common prefixes, like 'משה'
def strip_prefixes(token):
    t = token

    # First, check if the token itself is one of the known person names.
    # If so, return it directly, as it's not a prefixed form we want to strip.
    # This assumes 'person_names' is accessible in scope (defined below).
    if t in person_names:
        return t

    # Iteratively strip prefixes only if the current token starts with a prefix
    # and the resulting token is either a known name or the original token was not a known name.
    # Loop to handle cases with multiple prefixes (e.g., 'ובמשה')
    while len(t) > 1 and t[0] in prefixes:
        # Check if the current 't' (after some stripping) is now a known name.
        # If it is, this is the base form we want.
        if t in person_names:
            return t
        t = t[1:]

    return t # If no known name found after stripping, return the final stripped token

person_names = {
    "משה", "אהרן", "יהושע", "בצלאל", "אהליאב", "פרעה", "יוסף",
    "יעקב", "אברהם", "יצחק", "ראובן", "שמעון", "לוי", "יהודה",
    "יששכר", "זבולן", "בנימן", "דן", "נפתלי", "גד", "אשר", "איתמר"
}

counts = Counter()

for w in words:
    base = strip_prefixes(w)
    if base in person_names:
        counts[base] += 1

print(counts.most_common(20))

[('אשר', 310), ('משה', 291), ('פרעה', 116), ('אהרן', 116), ('יעקב', 11), ('לוי', 9), ('אברהם', 9), ('יצחק', 9), ('יהושע', 7), ('בצלאל', 6), ('אהליאב', 5), ('יהודה', 4), ('דן', 4), ('יוסף', 4), ('ראובן', 3), ('שמעון', 3), ('איתמר', 3), ('גד', 2), ('יששכר', 1), ('זבולן', 1)]


In [ ]:
from transformers import pipeline
from collections import Counter
import re

In [ ]:
# Hebrew NER pipeline
# You may need to swap the model if this one is unavailable in your environment.
ner = pipeline(
    "ner",
    model="avichr/heBERT_NER",
    aggregation_strategy="simple"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: avichr/heBERT_NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Use the text you already cleaned
sample_text = clean_text[:1500]   # start small first
entities = ner(sample_text)

entities[:20]

[{'entity_group': 'B_PERS',
  'score': np.float32(0.7569672),
  'word': 'שמעון לוי',
  'start': 61,
  'end': 70},
 {'entity_group': 'B_PERS',
  'score': np.float32(0.78681135),
  'word': '##ודה יששכר',
  'start': 74,
  'end': 83},
 {'entity_group': 'B_PERS',
  'score': np.float32(0.49657148),
  'word': '##ן',
  'start': 88,
  'end': 89},
 {'entity_group': 'B_PERS',
  'score': np.float32(0.57121307),
  'word': '##מן',
  'start': 94,
  'end': 96},
 {'entity_group': 'B_PERS',
  'score': np.float32(0.50522625),
  'word': '##סף',
  'start': 154,
  'end': 156}]

In [ ]:
# Keep only person entities
person_entities = []

for ent in entities:
    label = ent.get("entity_group", "")
    if label in {"PER", "PERSON"}:
        person_entities.append(ent["word"])

person_entities[:30]

[]

In [ ]:
# Normalize a little
def normalize_entity(x):
    x = re.sub(r"[^\u05D0-\u05EA\s]", "", x)  # keep Hebrew letters/spaces
    x = re.sub(r"\s+", " ", x).strip()
    return x

normalized_people = [normalize_entity(x) for x in person_entities if normalize_entity(x)]
person_counts = Counter(normalized_people)

person_counts.most_common(20)

[]

In [ ]:
def chunk_text(text, chunk_size=1500):
    words = text.split()
    for i in range(0, len(words), chunk_size):
        yield " ".join(words[i:i+chunk_size])

# Ensure strip_prefixes is defined or accessible
# It's already defined in cell yH40wRRrgHs8, but redefining here for clarity in this context
prefixes = ("ו", "ב", "ל", "כ", "מ", "ה")

def strip_prefixes(token):
    t = token
    while len(t) > 2 and t[0] in prefixes:
        t = t[1:]
    return t

all_people = []

for chunk in chunk_text(clean_text, chunk_size=300): # Further reduced chunk_size to avoid exceeding token limit
    entities = ner(chunk)
    for ent in entities:
        label = ent.get("entity_group", "")
        if 'PERS' in label: # Check for labels containing 'PERS' to capture B_PERS and I_PERS
            name = normalize_entity(ent["word"])
            # Apply prefix stripping after normalization for better consolidation
            if name:
                stripped_name = strip_prefixes(name)
                if stripped_name:
                    all_people.append(stripped_name)

ner_counts = Counter(all_people)
ner_counts.most_common(30)

[('אהרן', 11),
 ('שה', 4),
 ('ישראל', 4),
 ('צרים', 2),
 ('ראובן', 2),
 ('י לתל', 2),
 ('עמרם', 2),
 ('נדב', 2),
 ('קרח', 2),
 ('רן', 2),
 ('שמעון לוי', 1),
 ('דה יששכר', 1),
 ('ן', 1),
 ('ן דן', 1),
 ('סף', 1),
 ('יצחק וליעקב', 1),
 ('אלי', 1),
 ('פרעה', 1),
 ('ום', 1),
 ('אב', 1),
 ('פלוא', 1),
 ('רמ', 1),
 ('שמעון ימואל וימ', 1),
 ('אה', 1),
 ('יכין וצחר ושאול', 1),
 ('נית', 1),
 ('שמעון', 1),
 ('גרשון וקה', 1),
 ('חיי לוי', 1),
 ('שלשים', 1)]

### Re-evaluating Person Counts with a Rule-Based Approach

Given that the `avichr/heBERT_NER` model might be undercounting specific key figures or misclassifying some entities, let's use the rule-based approach (prefix stripping and dictionary lookup) that we used previously. This method often provides a more comprehensive count for known names in a text like this, as it doesn't rely on the nuanced contextual understanding of an NER model.

In [ ]:
from collections import Counter

# These definitions are already in cell yH40wRRrgHs8, re-defining for clarity in this new context
prefixes = ("ו", "ב", "ל", "כ", "מ", "ה")

# Modified strip_prefixes to handle names that start with common prefixes, like 'משה'
def strip_prefixes(token):
    t = token

    # First, check if the token itself is one of the known person names.
    # If so, return it directly, as it's not a prefixed form we want to strip.
    if t in person_names:
        return t

    # Iteratively strip prefixes only if the current token starts with a prefix
    # and the resulting token is either a known name or the original token was not a known name.
    # Loop to handle cases with multiple prefixes (e.g., 'ובמשה')
    while len(t) > 1 and t[0] in prefixes:
        # Check if the current 't' (after some stripping) is now a known name.
        # If it is, this is the base form we want.
        if t in person_names:
            return t
        t = t[1:]

    return t # If no known name found after stripping, return the final stripped token

person_names = {
    "משה", "אהרן", "יהושע", "בצלאל", "אהליאב", "פרעה", "יוסף",
    "יעקב", "אברהם", "יצחק", "ראובן", "שמעון", "לוי", "יהודה",
    "יששכר", "זבולן", "בנימן", "דן", "נפתלי", "גד", "אשר", "איתמר"
}

rule_based_counts = Counter()

# Assuming 'words' (tokenized clean_text) is available from previous cells
for w in words:
    base = strip_prefixes(w)
    if base in person_names:
        rule_based_counts[base] += 1

print("Top 20 person names using rule-based prefix stripping and dictionary lookup:")
print(rule_based_counts.most_common(20))

Top 20 person names using rule-based prefix stripping and dictionary lookup:
[('אשר', 310), ('משה', 291), ('פרעה', 116), ('אהרן', 116), ('יעקב', 11), ('לוי', 9), ('אברהם', 9), ('יצחק', 9), ('יהושע', 7), ('בצלאל', 6), ('אהליאב', 5), ('יהודה', 4), ('דן', 4), ('יוסף', 4), ('ראובן', 3), ('שמעון', 3), ('איתמר', 3), ('גד', 2), ('יששכר', 1), ('זבולן', 1)]


In [ ]:
import requests
import re
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# Ensure stopwords are downloaded for English
try:
    stopwords.words('english')
except OSError:
    print("Downloading NLTK English stopwords...")
    nltk.download('stopwords')

# 1. Fetch English text for Exodus from Sefaria API
english_ref = "Exodus" # Use the English name for the API call
# Requesting a specific English version, e.g., JPS 1985
english_data = requests.get(f"https://www.sefaria.org/api/v3/texts/{english_ref}", params={'version':'english'}).json()

# Extract the English text (flattening nested lists)
english_raw_text_list = english_data['versions'][0]['text']

def flatten_list(nested_list):
    flat_list = []
    for item in nested_list:
        if isinstance(item, list):
            flat_list.extend(flatten_list(item))
        else:
            flat_list.append(item)
    return flat_list

english_flat_text = ' '.join(flatten_list(english_raw_text_list))

# 2. Clean the English text
english_clean_text = re.sub(r'<.*?>', '', english_flat_text) # Remove HTML tags
english_clean_text = re.sub(r'[^a-zA-Z\s]', '', english_clean_text) # Keep only letters and spaces
english_clean_text = re.sub(r'\s+', ' ', english_clean_text).strip() # Replace multiple spaces
english_clean_text = english_clean_text.lower() # Convert to lowercase

# 3. Define English person names (extend as needed)
english_person_names = {
    "moses", "aaron", "joshua", "bezalel", "oholiab", "pharaoh", "joseph",
    "jacob", "abraham", "isaac", "reuben", "simeon", "levi", "judah",
    "issachar", "zebulun", "benjamin", "dan", "naphtali", "gad", "asher", "ithamar", "hur"
}

# 4. Tokenize, filter stopwords, and count English names
english_words = word_tokenize(english_clean_text)
english_stop_words = set(stopwords.words('english'))
filtered_english_words = [word for word in english_words if word not in english_stop_words and len(word) > 1]

english_counts = Counter()
for word in filtered_english_words:
    if word in english_person_names:
        english_counts[word] += 1

print("Top 20 person names in English (rule-based with NLTK):")
print(english_counts.most_common(20))

Top 20 person names in English (rule-based with NLTK):
[('moses', 286), ('aaron', 106), ('pharaoh', 88), ('abraham', 10), ('isaac', 10), ('jacob', 9), ('joshua', 7), ('hur', 6), ('bezalel', 6), ('levi', 5), ('oholiab', 5), ('judah', 4), ('dan', 4), ('joseph', 4), ('reuben', 3), ('simeon', 3), ('ithamar', 3), ('asher', 2), ('issachar', 1), ('zebulun', 1)]


### Combining AI-based NER with Rule-based Counting for 'משה'

Given the observed undercounting of 'משה' (Moses) by the general NER model, we'll implement a hybrid strategy. This approach will:
1. Start with the person counts identified by the NER model.
2. Manually count occurrences of 'משה' and its common prefixed forms directly from the `words` list (the tokenized Hebrew text).
3. Replace or add the manual 'משה' count into the combined results, ensuring a more accurate representation for this key figure.

In [ ]:
from collections import Counter

# Start with the NER counts as our base
hybrid_counts = ner_counts.copy()

# Define prefixes and strip_prefixes function again for clarity, though it's already defined.
# This ensures the logic is self-contained if cells are run out of order.
prefixes = ("ו", "ב", "ל", "כ", "מ", "ה")

# Modified strip_prefixes to handle names that start with common prefixes, like 'משה'
def strip_prefixes(token):
    t = token

    # First, check if the token itself is one of the known person names.
    # If so, return it directly, as it's not a prefixed form we want to strip.
    if t in person_names:
        return t

    # Iteratively strip prefixes only if the current token starts with a prefix
    # and the resulting token is either a known name or the original token was not a known name.
    # Loop to handle cases with multiple prefixes (e.g., 'ובמשה')
    while len(t) > 1 and t[0] in prefixes:
        # Check if the current 't' (after some stripping) is now a known name.
        # If it is, this is the base form we want.
        if t in person_names:
            return t
        t = t[1:]

    return t # If no known name found after stripping, return the final stripped token

# List of known person names (same as used in rule_based_counts)
person_names = {
    "משה", "אהרן", "יהושע", "בצלאל", "אהליאב", "פרעה", "יוסף",
    "יעקב", "אברהם", "יצחק", "ראובן", "שמעון", "לוי", "יהודה",
    "יששכר", "זבולן", "בנימן", "דן", "נפתלי", "גד", "אשר", "איתמר"
}

# Populate a temporary counter for rule-based counts for known names
temp_rule_based_counts = Counter()
for w in words:
    base = strip_prefixes(w)
    if base in person_names:
        temp_rule_based_counts[base] += 1

# Now, override/update the hybrid_counts with the more accurate rule-based counts for known names
for name, count in temp_rule_based_counts.items():
    hybrid_counts[name] = count

# Clean up any fragmented NER entries that correspond to a now-properly-counted known name.
# For example, if 'שה' was an NER output for 'משה', remove it now that 'משה' is counted.
if 'שה' in hybrid_counts and 'משה' in hybrid_counts:
    del hybrid_counts['שה']

# Additional specific fragment cleanups based on observed NER output
fragments_to_remove = [
    'שמעון לוי', 'דה יששכר', 'ן', 'ן דן', 'סף', 'יצחק וליעקב',
    'אלי', 'ום', 'אב', 'פלוא', 'רמ', 'שמעון ימואל וימ', 'אה', 'יכין וצחר ושאול',
    'נית', 'י לתל', 'גרשון וקה', 'חיי לוי', 'שלשים', 'גרשון', 'שמע', 'קה',
    'הר', 'ון', 'אל', 'חיי קה', 'שים', 'יוכ', 'דד', 'ד', 'חיי עמרם', 'יצהר קרח ונפג',
    'עזיאל מישאל', 'צפן וסתר'
]

for frag in fragments_to_remove:
    if frag in hybrid_counts and frag not in person_names:
        # Only remove if it's not a known person name itself (to avoid removing correct entities)
        del hybrid_counts[frag]

print("Top 20 person names using Hybrid (NER + targeted rule for known names) approach:")
print(hybrid_counts.most_common(20))

Top 20 person names using Hybrid (NER + targeted rule for known names) approach:
[('אשר', 310), ('משה', 291), ('פרעה', 116), ('אהרן', 116), ('יעקב', 11), ('לוי', 9), ('אברהם', 9), ('יצחק', 9), ('יהושע', 7), ('בצלאל', 6), ('אהליאב', 5), ('ישראל', 4), ('יהודה', 4), ('דן', 4), ('יוסף', 4), ('ראובן', 3), ('שמעון', 3), ('איתמר', 3), ('צרים', 2), ('עמרם', 2)]
